# Análisis de Datos Faltantes (Missing Data)

En esta sesión exploraremos las estrategias para identificar y tratar valores nulos en nuestros datasets. El manejo adecuado de datos faltantes es crucial, ya que muchos algoritmos de Machine Learning no pueden procesar valores `NaN` o `None` de forma nativa.

### Objetivos de la notebook:
1. Identificar patrones de datos faltantes mediante visualización.
2. Aplicar técnicas de imputación univariada (media, mediana, moda).
3. Implementar técnicas de imputación multivariada (KNN y MICE).
4. Aprender a utilizar indicadores de ausencia como variables adicionales.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer, MissingIndicator

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Carga de Datos

Utilizaremos el dataset de **House Prices** de Kaggle. Para asegurar que la notebook funcione en Google Colab, cargaremos los datos desde un repositorio público.

In [ ]:
# Descargar desde https://www.kaggle.com/competitions/home-data-for-ml-course/data?select=train.csv y subir a drive/colab
df = pd.read_csv('train.csv')
df.head()

## 2. Identificación y Visualización

Antes de imputar, debemos entender cuántos datos faltan y si hay algún patrón (ej. si los faltantes de una columna coinciden con los de otra).

In [ ]:
print("Valores nulos por columna (Top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False)
#plt.figure(figsize=(6, 10))
#sns.heatmap(df.isnull().T, cbar=False, xticklabels=False)
plt.title('Mapa de Calor de Datos Faltantes')
plt.show()

## 3. Imputación Univariada

La imputación univariada completa los valores faltantes de una columna utilizando únicamente los valores presentes en esa misma columna.

- **Media/Mediana**: Para variables numéricas.
- **Más frecuente (Moda)**: Para variables categóricas.

In [ ]:
df_a_imputar = df.copy()

imp_mean = SimpleImputer(strategy='mean')

print(f"Faltantes en Lot Frontage previa imputación: {df_a_imputar['LotFrontage'].isna().sum()}")
# Identificamos una columna numérica con nulos para el ejemplo
if 'LotFrontage' in df_a_imputar.columns:
    df_a_imputar[['LotFrontage']] = imp_mean.fit_transform(df_a_imputar[['LotFrontage']])

print(f"Faltantes en Lot Frontage tras imputación: {df_a_imputar['LotFrontage'].isna().sum()}")

In [ ]:
df.shape[0]

Estamos imputando 259 valores de un total de 1460 registros (~17%)

¿Que impacto tiene esto en la distribución del dataset? No deja de ser "inventar datos", despues de todo.

In [ ]:
# Imputamos por mediana
df_a_imputar_mediana = df.copy()
imp_median = SimpleImputer(strategy='median')
df_a_imputar_mediana[['LotFrontage']] = imp_median.fit_transform(df_a_imputar_mediana[['LotFrontage']])

In [ ]:
col = 'LotFrontage'

# Distribución Original (eliminando NaNs para el plot)
sns.kdeplot(df[col].dropna(), label='Original (Sin Nulos)', color='black', lw=3, ls='--')

# Distribución con Media
sns.kdeplot(df_a_imputar[col], label='Imputación por Media', color='red', alpha=0.6)

# Distribución con Mediana
sns.kdeplot(df_a_imputar_mediana[col], label='Imputación por Mediana', color='blue', alpha=0.6)

plt.title(f'Impacto de la Imputación Univariada en {col}', fontsize=15)
plt.xlabel('Pies de frente del lote')
plt.ylabel('Densidad')
plt.legend()
plt.show()

In [ ]:
print(f"Varianza original: {df[col].var():.2f}")
print(f"Varianza tras media: {df_a_imputar[col].var():.2f}")
print(f"Varianza tras mediana: {df_a_imputar_mediana[col].var():.2f}")

## 4. Imputación Multivariada

A diferencia de la univariada, estas técnicas utilizan la relación entre múltiples variables para estimar el valor faltante.

### 4.1 MICE (Iterative Imputer)
Modela cada característica con valores faltantes como una función de otras características.

In [ ]:
mice_imputer = IterativeImputer(
    max_iter=30,
    random_state=42,
    # initial_strategy='median'
)
# Seleccionamos algunas variables numéricas para el ejemplo
numeric_cols = df.select_dtypes(include=[np.number]).columns
df_mice = mice_imputer.fit_transform(df[numeric_cols])
print("Imputación MICE completada.")

In [ ]:
# posición de 'LotFrontage' en las columnas del df imputado
idx = list(numeric_cols).index('LotFrontage')

# Tomamos las columnas que queremos comparar
series_orig = df['LotFrontage'].dropna()
series_mean = df['LotFrontage'].fillna(df['LotFrontage'].mean())
series_mice = df_mice[:, idx]

# Visualización comparativa
plt.figure(figsize=(12, 7))
sns.kdeplot(series_orig, label='Original (Sin Nulos)', color='black', lw=2, ls='--')
sns.kdeplot(series_mean, label='Imputación por Media (Univariada)', color='red', alpha=0.5)
sns.kdeplot(series_mice, label='Imputación MICE (Multivariada)', color='green', alpha=0.8)
plt.title('Comparativa de Distribuciones: Original vs. Media vs. MICE', fontsize=14)
plt.xlabel('LotFrontage')
plt.ylabel('Densidad')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Miramos la varianza
print("Comparación de Desviación Estándar (Varianza):")
print(f"Original: {series_orig.std():.2f}")
print(f"Media:    {series_mean.std():.2f}")
print(f"MICE:     {series_mice.std():.2f}")

### 4.2 KNN Imputer
Utiliza los vecinos más cercanos (K-Nearest Neighbors) para encontrar registros similares e imputar el valor promedio de esos vecinos.

In [ ]:
knn_imputer = KNNImputer(n_neighbors=5)
df_knn = knn_imputer.fit_transform(df[numeric_cols])
print("Imputación KNN completada.")

In [ ]:
series_knn = df_knn[:, idx]

# Visualización comparativa
plt.figure(figsize=(12, 7))
sns.kdeplot(series_orig, label='Original (Sin Nulos)', color='black', lw=2, ls='--')
sns.kdeplot(series_mean, label='Imputación por Media (Univariada)', color='red', alpha=0.5)
sns.kdeplot(series_mice, label='Imputación MICE (Multivariada)', color='green', alpha=0.8)
sns.kdeplot(series_knn, label='Imputación KNN (Multivariada)', color='blue', alpha=0.8)
plt.title('Comparativa de Distribuciones: Original vs. Media vs. MICE', fontsize=14)
plt.xlabel('LotFrontage')
plt.ylabel('Densidad')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Miramos la varianza
print("Comparación de Desviación Estándar (Varianza):")
print(f"Original: {series_orig.std():.2f}")
print(f"Media:    {series_mean.std():.2f}")
print(f"MICE:     {series_mice.std():.2f}")
print(f"KNN:      {series_knn.std():.2f}")

## 5. Indicadores de Ausencia

A veces, el hecho de que falte un dato es en sí mismo una información valiosa (ej. un sensor que falla bajo ciertas condiciones).

In [ ]:
indicator = MissingIndicator()
absence_matrix = indicator.fit_transform(df)
print(f"Forma de la matriz de indicadores: {absence_matrix.shape}")